In [1]:
import nasdaqdatalink

nasdaqdatalink.read_key(r"key\secret.txt")

df = nasdaqdatalink.get_table(
    "WIKI/PRICES",
    ticker="AAPL",
    date={"gte": "1996-01-01", "lte": "2026-10-27"},
    paginate=True,
)
df = df.reset_index(drop=True)
df.head()

,ticker,date,open,high,low,close,volume,ex-dividend,split_ratio,adj_open,adj_high,adj_low,adj_close,adj_volume
0,AAPL,2018-03-27,173.68,175.15,166.92,168.340,38962839.0,0.0,1.0,173.68,175.15,166.92,168.340,38962839.0
1,AAPL,2018-03-26,168.07,173.10,166.44,172.770,36272617.0,0.0,1.0,168.07,173.10,166.44,172.770,36272617.0
2,AAPL,2018-03-23,168.39,169.92,164.94,164.940,40248954.0,0.0,1.0,168.39,169.92,164.94,164.940,40248954.0
3,AAPL,2018-03-22,170.00,172.68,168.60,168.845,41051076.0,0.0,1.0,170.00,172.68,168.60,168.845,41051076.0
4,AAPL,2018-03-21,175.04,175.09,171.26,171.270,35247358.0,0.0,1.0,175.04,175.09,171.26,171.270,35247358.0


In [2]:
import numpy as np

df = df.sort_values("date").reset_index(drop=True)

df["simple_return"] = df["adj_close"].pct_change()
df["log_return"] = np.log(df["adj_close"] / df["adj_close"].shift(1))

df[["date", "adj_close", "simple_return", "log_return"]].head()

,date,adj_close,simple_return,log_return
0,1996-01-02,1.032288,NaN,NaN
1,1996-01-03,1.032288,0.000000,0.000000
2,1996-01-04,1.013975,-0.017740,-0.017900
3,1996-01-05,1.100401,0.085234,0.081796
4,1996-01-08,1.112609,0.011095,0.011034


In [3]:
import pandas as pd

df_cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL")
df_cpi = df_cpi.reset_index(drop=True)
df_cpi.head()

,observation_date,CPIAUCSL
0,1947-01-01,21.48
1,1947-02-01,21.62
2,1947-03-01,22.00
3,1947-04-01,22.00
4,1947-05-01,21.95


In [7]:
import pandas as pd
import numpy as np

# 1. CPI fresh load (FRED)
df_cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL")
df_cpi.columns = ["date", "cpi"]
df_cpi["date"] = pd.to_datetime(df_cpi["date"])
df_cpi = df_cpi.set_index("date")

# 2. AAPL: monthly last adj_close (df = WIKI/PRICES data)
df["date"] = pd.to_datetime(df["date"])
monthly = df.set_index("date")["adj_close"].resample("MS").last().to_frame("adj_close")

# 3. Merge
data = monthly.join(df_cpi, how="inner")

# 4. Returns
data["simple_return"] = data["adj_close"].pct_change()
data["log_return"] = np.log(data["adj_close"] / data["adj_close"].shift(1))

# 5. Inflation rate (month-on-month)
data["inflation"] = data["cpi"].pct_change()

# 6. Real return
data["real_return"] = (1 + data["simple_return"]) / (1 + data["inflation"]) - 1
data = data.reset_index()
data.head()

,date,adj_close,cpi,simple_return,log_return,inflation,real_return
0,1996-01-01,0.887389,154.7,NaN,NaN,NaN,NaN
1,1996-02-01,0.883533,155.0,-0.004345,-0.004354,0.001939,-0.006272
2,1996-03-01,0.789076,155.5,-0.106909,-0.113067,0.003226,-0.109781
3,1996-04-01,0.782971,156.1,-0.007736,-0.007766,0.003859,-0.011550
4,1996-05-01,0.839196,156.4,0.071810,0.069348,0.001922,0.069754
